# 04 Supplementary Figures S1 to S3 and Supplementary Tables S3, S6, S7 (S5 cell builds the Word file with S1 to S4)
Fresh kernel, run S0 to S4 in order. Needs notebook 02 (annotated object and cell B5 tables) and notebook 03 (Z tables).
Figures go to `figures/supplement/` and `final_data/figures/`, tables to `final_data/supplementary_tables/`.

| Item | Content | First cited |
|---|---|---|
| Fig S1 | methods controls: A–C atlas registration, D–F home regions, G–I label transfer | Methods |
| Fig S2 | Spp1+ microglia: A Spp1 transcripts per cell, B–C identity control | Results 1 and 2 |
| Fig S3 | astrocyte association per animal | Results 3 |
| Table S1 | source data Figure 1 (notebook 01) | Results 1 |
| Table S2 | robustness, own cohort | Results 1 |
| Table S3 | source data Figure 2 (notebook 02) | Results 2 |
| Table S4 | transcript level, all public animals | Results 2 |
| Table S5 | source data Figure 3 (notebook 02) | Results 3 |
| Table S6 | source data Supplementary Figures S1 to S3 | Data availability |

In [ ]:
# ==========================================================
# S0) Supplement: setup, helpers, data (fresh kernel)
# ==========================================================
import numpy as np, pandas as pd, scanpy as sc, anndata as ad, warnings
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from PIL import Image
from scipy.spatial import cKDTree
from scipy.stats import fisher_exact
import statsmodels.api as sm, statsmodels.formula.api as smf
from openpyxl.styles import Font, Alignment
warnings.filterwarnings("ignore")

import os
# Project folder with data/, tables/, figures/ and final_data/ (see README).
# Default: the parent of the notebooks folder. Override with the environment variable SPP1_PROJECT_DIR.
OUT = Path(os.environ.get("SPP1_PROJECT_DIR", Path.cwd().parent)).resolve()
FIG, TAB, DAT = OUT / "figures", OUT / "tables", OUT / "data"
SUPP = FIG / "supplement"; SUPP.mkdir(parents=True, exist_ok=True)
XLS = OUT / "final_data" / "supplementary_tables"; XLS.mkdir(parents=True, exist_ok=True)
if "C" in globals() and not callable(C): del C

mpl.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
                     "font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7, "xtick.labelsize": 6.5,
                     "ytick.labelsize": 6.5, "legend.fontsize": 6.5, "axes.linewidth": 0.6, "pdf.fonttype": 42,
                     "svg.fonttype": "none", "savefig.dpi": 300, "figure.dpi": 150, "legend.frameon": False,
                     "axes.spines.top": False, "axes.spines.right": False})
MM = 1 / 25.4
C_MG, C_POS, C_NEG, C_AS, C_FIB, C_BG = "#e0a100", "#c2185b", "#f1d48a", "#3b5bdb", "#9fb3c8", "#ececec"
C_WT, C_TG, C_YOUNG = "#3b5bdb", "#7a3e9d", "#a9b8ea"
CM_DOT = LinearSegmentedColormap.from_list("dot", ["#f1f3fb", "#9aa8dc", "#2b3f9e"])
OWN = ["SPF_Coronar", "AQP4KO_sagital", "Thy_CTRL"]
LAB = {"SPF_Coronar": "WT 3 mo", "AQP4KO_sagital": "AQP4 KO 3 mo", "Thy_CTRL": "WT 12 mo"}
# old sample labels (still used inside the Z tables of notebook 03) -> display labels
REN = {"SPF 12 wk": "WT 3 mo", "AQP4 KO 12 wk": "AQP4 KO 3 mo", "Healthy 1 y": "WT 12 mo", "healthy 1 y": "WT 12 mo",
       "1-year brain": "WT 12 mo section", "12 weeks": "3 months", "1 year": "12 months"}
def relabel(x):
    """Replace old sample labels in a string or in a DataFrame (cells, column names, index)."""
    if isinstance(x, pd.DataFrame):
        x = x.rename(columns=relabel, index=relabel)
        return x.apply(lambda c: c if pd.api.types.is_numeric_dtype(c) else c.map(relabel))
    if isinstance(x, str):
        for a, b in REN.items(): x = x.replace(a, b)
    return x
sig = lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
SD = {}                                   # source data of the supplementary figures -> Supplementary Table S6


FINAL = OUT / "final_data" / "figures"; FINAL.mkdir(parents=True, exist_ok=True)


def save_supp(fig, name):
    for d in [SUPP, FINAL]:
        fig.savefig(d / f"{name}.pdf", bbox_inches="tight"); fig.savefig(d / f"{name}.png", dpi=300, bbox_inches="tight")
        if d == FINAL:                                   # Frontiers: TIFF, RGB, 300 dpi, LZW
            Image.open(d / f"{name}.png").convert("RGB").save(d / f"{name}.tif", compression="tiff_lzw", dpi=(300, 300))
    plt.show()


def write_book(path, title, sheets):
    """Excel workbook: README sheet (title + list of sheets), then one sheet per panel in the given order.
    sheets = list of (sheet name, one-line description, DataFrame)."""
    readme = pd.DataFrame([(n, relabel(d), len(df)) for n, d, df in sheets], columns=["Sheet", "Content", "Rows"])
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        readme.to_excel(xw, sheet_name="README", index=False, startrow=2)
        for n, d, df in sheets:
            relabel(df).to_excel(xw, sheet_name=str(n)[:31], index=False, startrow=2)
        wb = xw.book
        wb["README"]["A1"] = title; wb["README"]["A1"].font = Font(bold=True, size=12)
        for n, d, df in sheets:
            ws = wb[str(n)[:31]]; ws["A1"] = relabel(d); ws["A1"].font = Font(bold=True)
        for ws in wb.worksheets:
            for c in ws[3]: c.font = Font(bold=True)
            ws.freeze_panes = "A4"
            for col in ws.iter_cols(min_row=3, max_row=min(ws.max_row, 200)):
                w = max(len(str(c.value)) if c.value is not None else 0 for c in col)
                ws.column_dimensions[col[0].column_letter].width = min(max(9, w + 2), 60)
        wb["README"].column_dimensions["B"].width = 110
    print("written:", path.name, "|", ", ".join(n for n, _, _ in sheets))


def boot_ci(x, n=2000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    b = np.random.default_rng(seed).choice(x, (n, len(x))).mean(1)
    return x.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)


def counts_of(adx, genes):
    gi = [adx.var_names.get_loc(g) for g in genes]
    M = adx.layers["counts"][:, gi]
    return M.toarray() if hasattr(M, "toarray") else np.asarray(M)


# own cohort (247-gene panel)
own = sc.read_h5ad(DAT / "own_cohort_xenium_247panel.h5ad")
own = own[own.obs["sample"].isin(OWN)].copy()
own.obs["x"], own.obs["y"] = np.asarray(own.obsm["spatial"])[:, 0], np.asarray(own.obsm["spatial"])[:, 1]
own.obs["Spp1_n"] = counts_of(own, ["Spp1"]).ravel()
# public 10x cohort (247 + 99 add-on genes), annotated in the 10xAD notebook (cell 4)
xa = sc.read_h5ad(DAT / "10xAD_all_annotated.h5ad")
ox = xa.obs
ox["age_mo"] = ox["age_mo"].astype(float)
if "total_counts" not in ox: ox["total_counts"] = ox["n_counts"]          # transcript depth per cell
ox["ctx_bin"] = pd.cut(ox["f_oligo"], [-0.01, 0.2, 0.4, 1.0], labels=["Grey", "Border", "Fibre"]).astype(str)
ox["animal"] = ox["genotype"].astype(str) + " " + ox["age_mo"].map(lambda a: f"{a:g}") + " mo"
ANIMALS = (ox[["genotype", "age_mo", "animal", "sample"]].drop_duplicates()
             .assign(o=lambda t: (t["genotype"] != "WT").astype(int)).sort_values(["o", "age_mo"]))
print(own.obs["sample"].value_counts().to_string()); print(ANIMALS[["animal", "sample"]].to_string(index=False))

In [ ]:
# ==========================================================
# S1) Supplementary Figure S1: methods controls (atlas registration, home regions, label transfer)
# ==========================================================
STRUCT = {"CP": "#4C72B0", "GPe": "#c2185b", "IC": "#e0a100", "arb": "#2e8b57"}
NAMES = {"CP": "caudoputamen", "GPe": "globus pallidus ext.", "IC": "internal capsule", "arb": "arbor vitae"}
NOT_HOME = ["White matter", "Ventricle / CP"]
HOME_COL = {"Cortex": "#a9b9dc", "Hippocampus": "#ecb48f", "Striatum": "#a8d29c", "Pallidum": "#86c5bc",
            "Thalamus": "#d9a7cc", "Hypothalamus": "#ead07c", "Midbrain": "#b8a6dc", "Hindbrain": "#c9bfa6",
            "Cerebellum": "#9cc7e0", "Olfactory": "#cccccc"}
CT_ORDER = ["Microglia", "Astrocyte", "Oligodendrocyte", "OPC", "Endothelial", "Mural", "VLMC / fibroblast",
            "Cortical excitatory", "Subcortical glutamat.", "Inhibitory neuron", "Striatal MSN", "Dentate granule"]
PLANE = {"SPF_Coronar": "coronal", "AQP4KO_sagital": "sagittal", "Thy_CTRL": "sagittal"}


def home_region(xy_q, xy_ref, reg_ref, ks=(100, 300, 1000), chunk=20000):
    """Majority grey-matter region among the k nearest cells (fibre tracts and ventricles inherit the region they traverse)."""
    ok = ~pd.Series(reg_ref).isin(NOT_HOME + ["nan"]).to_numpy() & pd.notna(reg_ref)
    cats = np.unique(reg_ref[ok]); code = np.full(len(reg_ref), -1); code[ok] = np.searchsorted(cats, reg_ref[ok])
    tree, out, todo = cKDTree(xy_ref), np.full(len(xy_q), -1), np.arange(len(xy_q))
    for k in ks:
        k = min(k, len(xy_ref)); left = []
        for s0 in range(0, len(todo), chunk):
            t = todo[s0:s0 + chunk]; c = code[tree.query(xy_q[t], k=k)[1]]
            cnt = np.stack([(c == j).sum(1) for j in range(len(cats))], 1); hit = cnt.sum(1) > 0
            out[t[hit]] = cnt[hit].argmax(1); left.append(t[~hit])
        todo = np.concatenate(left) if left else np.array([], int)
        if len(todo) == 0: break
    return np.where(out >= 0, cats[np.clip(out, 0, None)], "unassigned")


def section_axes(ax, oo, title):
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    ax.plot([x1 - 1100, x1 - 100], [y0 - 150] * 2, color="#222222", lw=1.0)
    ax.text(x1 - 600, y0 - 380, "1 mm", ha="center", fontsize=5.5)
    ax.set_title(title, loc="left", fontsize=7, fontweight="bold")


# ---- data: home region of every cell (own cohort)
own.obs["home"] = "unassigned"
for smp in OWN:
    m = (own.obs["sample"] == smp).to_numpy(); xy = own.obs.loc[m, ["x", "y"]].to_numpy()
    own.obs.loc[m, "home"] = home_region(xy, xy, own.obs.loc[m, "region_coarse"].astype(str).to_numpy())

# ---- data: label transfer (public cohort), add-on genes not used for training
ADDON = [g for g in ["P2ry12", "Tmem119", "Cx3cr1", "Gpr34", "Olfml3", "Aldh1l1", "Gja1", "Mobp", "Olig2", "Vcan"]
         if g in xa.var_names and g not in own.var_names]
cts = [c for c in CT_ORDER if c in set(ox["cell_type"].astype(str))]
Xa = counts_of(xa, ADDON); ctv = ox["cell_type"].astype(str).to_numpy(); dep_x = ox["total_counts"].to_numpy().astype(float)
det_t = pd.DataFrame([(Xa[ctv == c] > 0).mean(0) * 100 for c in cts], index=cts, columns=ADDON)
lg_t = pd.DataFrame([np.log1p(Xa[ctv == c] / dep_x[ctv == c][:, None] * 100).mean(0) for c in cts], index=cts, columns=ADDON)
lg_ts = (lg_t - lg_t.min()) / (lg_t.max() - lg_t.min()).replace(0, 1)
cb_ = pd.cut(ox["ct_conf"].astype(float), [-0.01, 0.5, 0.7, 1.01], labels=["< 0.5 (unassigned)", "0.5 to 0.7", "≥ 0.7"])
conf_t = pd.crosstab(ox["animal"], cb_, normalize="index").reindex(ANIMALS["animal"]) * 100
strict_t = ox.groupby("animal").agg(predicted_microglia=("cell_type", lambda v: int((v == "Microglia").sum())),
                                    strict_microglia=("is_mg", "sum"),
                                    predicted_astrocytes=("cell_type", lambda v: int((v == "Astrocyte").sum())),
                                    strict_astrocytes=("is_astro", "sum")).reindex(ANIMALS["animal"])

# ---- figure
fig = plt.figure(figsize=(180 * MM, 215 * MM))
G = fig.add_gridspec(3, 1, height_ratios=[1, 1, 1.25], hspace=0.28)
reg_rows, home_rows = [], []
gA = G[0].subgridspec(1, 3, wspace=0.05)
for k, smp in enumerate(OWN):                                           # A-C registration check
    ax = fig.add_subplot(gA[k]); oo = own.obs[own.obs["sample"] == smp]; st = oo["region_struct_smooth"].astype(str)
    ax.scatter(oo["x"], oo["y"], s=0.05, c=C_BG, lw=0, rasterized=True)
    for s_, c_ in STRUCT.items():
        m = st == s_
        ax.scatter(oo.loc[m, "x"], oo.loc[m, "y"], s=0.25, c=c_, lw=0, rasterized=True)
        for reg, n in oo.loc[m, "region_coarse"].astype(str).value_counts().items():
            reg_rows.append({"section": LAB[smp], "atlas structure": s_, "coarse region of the cells": reg, "cells": int(n)})
    section_axes(ax, oo, f"# S1{'ABC'[k]}  {LAB[smp]} ({PLANE[smp]})")
    if k == 2:
        for s_, c_ in STRUCT.items(): ax.scatter([], [], s=12, c=c_, label=f"{s_}  {NAMES[s_]}")
        ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=5.5, handletextpad=0.2, title="atlas label", title_fontsize=5.5)
gD = G[1].subgridspec(1, 3, wspace=0.05)
for k, smp in enumerate(OWN):                                           # D-F home regions
    ax = fig.add_subplot(gD[k]); oo = own.obs[own.obs["sample"] == smp]
    ax.scatter(oo["x"], oo["y"], s=0.06, c=oo["home"].map(HOME_COL).fillna("#eeeeee"), lw=0, rasterized=True)
    pp = oo[(oo["cell_type"] == "Microglia") & (oo["Spp1_n"] >= 1)]
    ax.scatter(pp["x"], pp["y"], s=4, c=C_POS, lw=0.2, edgecolor="white", zorder=5, rasterized=True)
    mgo = oo[oo["cell_type"] == "Microglia"]
    t = mgo.groupby("home").agg(microglia=("Spp1_n", "size"), spp1_pos=("Spp1_n", lambda v: int((v >= 1).sum())))
    t["pct_spp1_pos"] = 100 * t["spp1_pos"] / t["microglia"]; t["cells"] = oo["home"].value_counts().reindex(t.index)
    home_rows.append(t.reset_index().rename(columns={"home": "home region"}).assign(section=LAB[smp]))
    section_axes(ax, oo, f"# S1{'DEF'[k]}  {LAB[smp]}, home regions")
    if k == 2:
        for r_, c_ in HOME_COL.items():
            if (own.obs["home"] == r_).any(): ax.scatter([], [], s=14, c=c_, marker="s", label=r_)
        ax.scatter([], [], s=10, c=C_POS, label="Spp1+ microglia")
        ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=5.5, handletextpad=0.2)
gG = G[2].subgridspec(1, 3, width_ratios=[2.3, 1.1, 0.9], wspace=0.75)
axG = fig.add_subplot(gG[0])                                            # G add-on genes by transferred cell type
for i, c in enumerate(cts):
    for j, g in enumerate(ADDON):
        axG.scatter(j, i, s=3 + 0.9 * det_t.loc[c, g], c=[CM_DOT(lg_ts.loc[c, g])], edgecolor="#555555", lw=0.3)
axG.set_xticks(range(len(ADDON))); axG.set_xticklabels(ADDON, rotation=60, ha="right", style="italic")
axG.set_yticks(range(len(cts))); axG.set_yticklabels(cts); axG.set_ylim(len(cts) - 0.5, -0.5)
axG.tick_params(length=0); [s_.set_visible(False) for s_ in axG.spines.values()]
for v in [25, 50, 75]: axG.scatter([], [], s=3 + 0.9 * v, c="#cccccc", edgecolor="#555555", lw=0.3, label=f"{v}%")
axG.legend(loc="upper left", bbox_to_anchor=(0.0, -0.2), ncol=3, fontsize=5.5)
axG.set_title("# S1G  Add-on genes by transferred cell type", loc="left", fontsize=7, fontweight="bold")
axH = fig.add_subplot(gG[1]); left = np.zeros(len(conf_t))             # H transfer confidence
for col, c_ in zip(conf_t.columns, ["#d9d9d9", "#9aa8dc", "#2b3f9e"]):
    axH.barh(range(len(conf_t)), conf_t[col], left=left, color=c_, label=col, height=0.7); left += conf_t[col].to_numpy()
axH.set_yticks(range(len(conf_t))); axH.set_yticklabels(conf_t.index); axH.invert_yaxis(); axH.set_xlabel("Cells (%)")
axH.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), fontsize=5.3, title="max. class probability", title_fontsize=5.5)
axH.set_title("# S1H  Transfer confidence", loc="left", fontsize=7, fontweight="bold")
axI = fig.add_subplot(gG[2]); yy = np.arange(len(strict_t))            # I strict cells per animal
axI.barh(yy - 0.18, strict_t["strict_microglia"], height=0.34, color=C_MG, label="strict microglia")
axI.barh(yy + 0.18, strict_t["strict_astrocytes"] / 4, height=0.34, color=C_AS, label="strict astrocytes / 4")
axI.set_yticks(yy); axI.set_yticklabels([]); axI.invert_yaxis(); axI.set_xlabel("Cells")
axI.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), fontsize=5.3)
axI.set_title("# S1I  Strict cells", loc="left", fontsize=7, fontweight="bold")
save_supp(fig, "FigS1_methods_controls")

SD["S1A-C"] = ("Fig S1A-C: cells carrying each atlas structure label, tabulated by the coarse region they lie in", pd.DataFrame(reg_rows))
SD["S1D-F"] = ("Fig S1D-F: cells, microglia and Spp1+ microglia per home region and section",
               pd.concat(home_rows, ignore_index=True)[["section", "home region", "cells", "microglia", "spp1_pos", "pct_spp1_pos"]].round(3))
SD["S1G_detection"] = ("Fig S1G: % of cells detecting each add-on gene (not used for label transfer), by transferred cell type, all six animals",
                       det_t.round(2).reset_index(names="cell type"))
SD["S1G_expression"] = ("Fig S1G: mean log-normalised expression (100 transcripts per cell) of the same genes", lg_t.round(4).reset_index(names="cell type"))
SD["S1H"] = ("Fig S1H: % of cells per animal by maximum class probability of the label transfer", conf_t.round(2).reset_index())
SD["S1I"] = ("Fig S1I: predicted and strict microglia and astrocytes per animal", strict_t.reset_index())
display(pd.DataFrame(reg_rows).pivot_table(index=["atlas structure", "coarse region of the cells"], columns="section", values="cells", fill_value=0))

In [ ]:
# ==========================================================
# S2) Supplementary Figure S2: Spp1+ microglia (Spp1 transcripts per cell, identity control)
# ==========================================================
BINS = [("1", 1, 1), ("2", 2, 2), ("3", 3, 3), ("≥4", 4, np.inf)]
G_MG = ["P2ry12", "Tmem119", "Gpr34", "Olfml3", "Siglech", "Cx3cr1", "Hexb", "Cd68"]
G_VASC = ["Acta2", "Cspg4", "Dcn", "Col1a1", "Igf2", "Cldn5", "Pecam1"]
REF = ["VLMC / fibroblast", "Mural", "Endothelial"]

# ---- data A: Spp1 transcripts per Spp1+ microglia, all nine animals
rows = [("Own cohort", LAB[s], own.obs.loc[(own.obs["sample"] == s) & (own.obs["cell_type"] == "Microglia"), "Spp1_n"].to_numpy())
        for s in OWN]
rows += [("Public cohort", r["animal"], ox.loc[(ox["sample"] == r["sample"]) & ox["is_mg"].astype(bool), "Spp1_n"].to_numpy())
         for _, r in ANIMALS.iterrows()]
spp = []
for coh, lab, v in rows:
    p = v[v >= 1]
    d = {"cohort": coh, "animal": lab, "microglia": len(v), "spp1_pos": len(p), "pct_spp1_pos": 100 * len(p) / max(len(v), 1),
         "mean_transcripts_per_pos": p.mean() if len(p) else np.nan}
    for b, lo, hi in BINS: d[f"pct_pos_with_{b}"] = 100 * ((p >= lo) & (p <= hi)).mean() if len(p) else np.nan
    spp.append(d)
spp = pd.DataFrame(spp)


# ---- data B, C: identity control
def identity_table(adx, obs, groups, genes):
    genes = [g for g in genes if g in adx.var_names]
    X = counts_of(adx, genes); dep = obs["total_counts"].to_numpy().astype(float); det, ex, n = {}, {}, {}
    for lab, m in groups:
        det[lab] = (X[m] > 0).mean(0) * 100; ex[lab] = np.log1p(X[m] / dep[m, None] * 100).mean(0); n[lab] = int(m.sum())
    return pd.DataFrame(det, index=genes).T, pd.DataFrame(ex, index=genes).T, n


def dotplot(ax, det, ex, title, n):
    exs = (ex - ex.min()) / (ex.max() - ex.min()).replace(0, 1)
    for i, r in enumerate(det.index):
        for j, g in enumerate(det.columns):
            ax.scatter(j, i, s=3 + 0.9 * det.loc[r, g], c=[CM_DOT(exs.loc[r, g])], edgecolor="#555555", lw=0.3)
    ax.set_xticks(range(det.shape[1])); ax.set_xticklabels(det.columns, rotation=60, ha="right", style="italic")
    ax.set_yticks(range(det.shape[0])); ax.set_yticklabels([f"{r} (n = {n[r]})" for r in det.index])
    ax.set_ylim(det.shape[0] - 0.5, -0.5); ax.tick_params(length=0); [s_.set_visible(False) for s_ in ax.spines.values()]
    ax.axvline(sum(g in G_MG for g in det.columns) - 0.5, color="#cccccc", lw=0.6); ax.axhline(1.5, color="#cccccc", lw=0.6)
    ax.set_title(title, loc="left", fontsize=7, fontweight="bold")


s_wt = ANIMALS.loc[ANIMALS["genotype"] == "WT"].sort_values("age_mo")["sample"].iloc[-1]
m_s = (ox["sample"] == s_wt).to_numpy(); mgm = ox["is_mg"].astype(bool).to_numpy(); sp = (ox["Spp1_n"] >= 1).to_numpy()
ctx_ = ox["cell_type"].astype(str).to_numpy()
dB, eB, nB = identity_table(xa, ox, [("Spp1+ microglia", m_s & mgm & sp), ("Spp1− microglia", m_s & mgm & ~sp)] +
                            [(r, m_s & (ctx_ == r)) for r in REF], G_MG + G_VASC)
oo = own.obs; m_o = (oo["sample"] == "Thy_CTRL").to_numpy(); cto = oo["cell_type"].astype(str).to_numpy(); spo = (oo["Spp1_n"] >= 1).to_numpy()
dC, eC, nC = identity_table(own, oo, [("Spp1+ microglia", m_o & (cto == "Microglia") & spo), ("Spp1− microglia", m_o & (cto == "Microglia") & ~spo)] +
                            [(r, m_o & (cto == r)) for r in REF], G_MG + G_VASC)

# ---- figure
fig = plt.figure(figsize=(180 * MM, 150 * MM))
G = fig.add_gridspec(2, 1, height_ratios=[1, 1.1], hspace=0.55)
gA = G[0].subgridspec(1, 2, width_ratios=[1.6, 1], wspace=0.1)
ax = fig.add_subplot(gA[0]); yy = np.arange(len(spp)); left = np.zeros(len(spp))
for (b, _, _), c_ in zip(BINS, ["#f3c3d6", "#e27ba3", "#c2185b", "#7a0e3a"]):
    vals = spp[f"pct_pos_with_{b}"].fillna(0).to_numpy()
    ax.barh(yy, vals, left=left, color=c_, height=0.7, label=f"{b} transcript{'s' if b != '1' else ''}"); left += vals
for y_, (_, r) in zip(yy, spp.iterrows()):
    ax.text(102, y_, f"{r['mean_transcripts_per_pos']:.1f}   (n = {r['spp1_pos']})", va="center", fontsize=5.8)
ax.text(102, -1.0, "mean per Spp1+ cell", fontsize=5.8, fontweight="bold")
ax.axhline(2.5, color="#999999", lw=0.6)
ax.set_yticks(yy); ax.set_yticklabels(spp["animal"]); ax.invert_yaxis(); ax.set_xlim(0, 100); ax.set_xlabel("Spp1+ microglia (%)")
ax.legend(loc="upper left", bbox_to_anchor=(1.32, 0.9), fontsize=5.5)
ax.set_title("# S2A  Spp1 transcripts per Spp1+ microglia", loc="left", fontsize=7, fontweight="bold", pad=12)
gB = G[1].subgridspec(1, 2, width_ratios=[dB.shape[1], dC.shape[1]], wspace=0.75)
dotplot(fig.add_subplot(gB[0]), dB, eB, "# S2B  Identity, public cohort, WT 13.4 mo", nB)
axC = fig.add_subplot(gB[1]); dotplot(axC, dC, eC, "# S2C  Identity, own cohort, WT 12 mo", nC)
for v in [25, 50, 75]: axC.scatter([], [], s=3 + 0.9 * v, c="#cccccc", edgecolor="#555555", lw=0.3, label=f"{v}%")
axC.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=5.5, labelspacing=1.0)
save_supp(fig, "FigS2_spp1_microglia")

SD["S2A"] = ("Fig S2A: Spp1+ microglia per animal and distribution of Spp1 transcripts per Spp1+ microglia "
             "(own cohort: annotated microglia, public cohort: strict microglia)", spp.round(3))
for k_, (d_, e_, lab_) in {"S2B": (dB, eB, "public cohort, WT 13.4 mo"), "S2C": (dC, eC, "own cohort, WT 12 mo")}.items():
    SD[f"{k_}_detection"] = (f"Fig {k_}: % of cells detecting each gene ({lab_})", d_.round(2).reset_index(names="group"))
    SD[f"{k_}_expression"] = (f"Fig {k_}: mean log-normalised expression, 100 transcripts per cell ({lab_})", e_.round(4).reset_index(names="group"))
display(spp.round(2)); display(dB.round(0)); display(dC.round(0))

In [ ]:
# ==========================================================
# S3) Supplementary Figure S3: astrocyte association per animal (all animals with at least ten Spp1+ microglia)
# ==========================================================
def grp_col(lab):
    return C_TG if "TgCRND8" in lab else C_WT if lab in ("WT 12 mo", "WT 13.4 mo") else C_YOUNG


# A: cell level, OR per astrocyte neighbour within 25 µm
z1 = pd.read_csv(TAB / "Z1_age_x_astrocyte_interaction.csv", index_col=0)
z1 = z1[(z1.index == "all three sections | n_astro25") & z1["model"].str.startswith("slope")]
cell = [{"cohort": "Own cohort", "animal": relabel(m.replace("slope ", "")), "OR": r["OR"], "lo": r["lo"], "hi": r["hi"], "p": r["p"],
         "n_spp1_pos": int(r["n_pos"]), "covariates": "context + home region"} for m, r in zip(z1["model"], z1.to_dict("records"))]
ox["n_astro25"] = np.nan
for s in ox["sample"].unique():
    m = np.where((ox["sample"] == s).to_numpy())[0]; xy = ox.iloc[m][["x_centroid", "y_centroid"]].to_numpy()
    mg_, as_ = ox["is_mg"].to_numpy()[m].astype(bool), ox["is_astro"].to_numpy()[m].astype(bool)
    ox.iloc[m[mg_], ox.columns.get_loc("n_astro25")] = cKDTree(xy[as_]).query_ball_point(xy[mg_], 25, return_length=True)
for _, a in ANIMALS.iterrows():
    d = ox[(ox["sample"] == a["sample"]) & ox["is_mg"].astype(bool)].copy(); d["spp1"] = (d["Spp1_n"] >= 1).astype(int)
    if d["spp1"].sum() < 10: continue
    f = smf.glm("spp1 ~ n_astro25 + C(ctx_bin)", data=d, family=sm.families.Binomial()).fit(cov_type="HC1")
    lo, hi = np.exp(f.conf_int().loc["n_astro25"])
    cell.append({"cohort": "Public cohort", "animal": a["animal"], "OR": np.exp(f.params["n_astro25"]), "lo": lo, "hi": hi,
                 "p": f.pvalues["n_astro25"], "n_spp1_pos": int(d["spp1"].sum()), "covariates": "context"})
cell = pd.DataFrame(cell)
# B, C: transcript level (public cohort only, from 10xAD cell 12)
tr = pd.read_csv(TAB / "10x_12_transcript_level_astro.csv")
tr = tr[tr["variant"] == "all"].copy()
tr["animal"] = tr["genotype"].astype(str) + " " + tr["age"].map(lambda a: f"{a:g}") + " mo"
order_tx = [a for a in ANIMALS["animal"] if a in set(tr["animal"])]
cov = tr[tr["gene"].str.startswith("coverage")].set_index("animal").reindex(order_tx)
gfp = tr[tr["gene"] == "Gfap"].set_index("animal").reindex(order_tx)

fig, axs = plt.subplots(1, 3, figsize=(180 * MM, 62 * MM), gridspec_kw={"wspace": 0.9, "width_ratios": [1.2, 1, 1]})
for ax, t, lab, ttl in [(axs[0], cell.set_index("animal"), "OR per astrocyte neighbour\n(cell level, 25 µm)", "# S3A  Cell level"),
                        (axs[1], cov, "Rate ratio, homeostatic astrocyte\ntranscripts (5 to 25 µm)", "# S3B  Astrocyte coverage"),
                        (axs[2], gfp, "OR, Gfap per homeostatic\nastrocyte transcript", "# S3C  Gfap enrichment")]:
    yy = np.arange(len(t)); cols = [grp_col(a) for a in t.index]
    ax.axvline(0, color="#999999", lw=0.6, ls="--")
    ax.hlines(yy, np.log2(t["lo"]), np.log2(t["hi"]), color=cols, lw=1.0)
    est = t["OR"] if "OR" in t.columns else t["ratio"]
    ax.scatter(np.log2(est), yy, s=22, c=cols, zorder=3)
    ax.set_yticks(yy); ax.set_yticklabels(t.index); ax.set_ylim(len(t) - 0.5, -0.5)
    ax.set_xlabel("log2 " + lab); ax.set_title(ttl, loc="left", fontsize=7, fontweight="bold")
    if "cohort" in t.columns and (t["cohort"] == "Own cohort").any():
        ax.axhline((t["cohort"] == "Own cohort").sum() - 0.5, color="#cccccc", lw=0.6)
for lab_, c_ in [("WT and AQP4 KO, 2.5 to 5.7 mo", C_YOUNG), ("WT 12 and 13.4 mo", C_WT), ("TgCRND8, all ages", C_TG)]:
    axs[2].scatter([], [], s=14, c=c_, label=lab_)
axs[2].legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=5.5)
save_supp(fig, "FigS3_per_animal")
SD["S3A"] = ("Fig S3A: odds ratio of the Spp1+ state per astrocyte neighbour within 25 µm, per animal (binomial GLM, HC1)", cell.round(4))
SD["S3B"] = ("Fig S3B: rate ratio of homeostatic astrocyte transcripts 5 to 25 µm around Spp1+ vs Spp1− microglia, per animal",
             cov.reset_index()[["animal", "ratio", "lo", "hi", "p", "q", "n_pos", "n_neg"]].round(4))
SD["S3C"] = ("Fig S3C: odds ratio of Gfap per homeostatic astrocyte transcript around Spp1+ vs Spp1− microglia, per animal",
             gfp.reset_index()[["animal", "ratio", "lo", "hi", "p", "q", "n_pos", "n_neg"]].round(4))
display(cell.round(3)); display(cov[["ratio", "lo", "hi", "q"]].round(3))

In [ ]:
# ==========================================================
# S4) Supplementary Tables S3, S6 and S7 (Excel, one sheet per panel or analysis, README first)
# ==========================================================
# --- Table S2: robustness of the astrocyte association, own cohort (from the Z notebook)
z1 = pd.read_csv(TAB / "Z1_age_x_astrocyte_interaction.csv").rename(columns={"Unnamed: 0": "subset | astrocyte variable"})
z4g = pd.read_csv(TAB / "Z4_S1_sensitivity_grid.csv")
z4l = pd.read_csv(TAB / "Z4_S1_leave_one_region_out.csv").rename(columns={"Unnamed: 0": "left out"})
z3 = pd.read_csv(TAB / "Z3_niche_composition.csv")
z5 = pd.read_csv(TAB / "Z5_astro_coverage_fibre_vs_grey.csv").rename(columns={"Unnamed: 0": "section"})
z2 = pd.read_csv(TAB / "Z2_astro_near_spp1_shell_radius_controls.csv")
write_book(XLS / "Supplementary_Table_S3_robustness_own_cohort.xlsx",
           "Supplementary Table S3. Robustness of the astrocyte association in our cohort",
           [("A_interaction", "A: age x astrocyte interaction and per-section slopes (binomial GLM, HC1; OR < 1 = fewer astrocytes around Spp1+ microglia)", z1),
            ("B_radius_threshold", "B: 1-year brain, OR per astrocyte neighbour by radius, Spp1 threshold and subset (context and home region as fixed effects)", z4g),
            ("C_leave_one_region", "C: 1-year brain, leave one home region out (25 µm, ≥1 Spp1 transcript)", z4l),
            ("D_composition", "D: neighbourhood composition around Spp1+ vs Spp1− microglia of the same section, context and home region (ratio, permutation p)", z3),
            ("E_coverage_by_context", "E: astrocyte coverage of Spp1− microglia, border/fibre vs grey context within home region", z5),
            ("F_astrocyte_profiles", "F: astrocytes next to Spp1+ vs Spp1− microglia: detection, mean log-normalised expression, fold change, permutation p (context x home region), all radii, contact excluded, homeostatic controls", z2)])

# --- Table S4: transcript-level astrocyte signal, public cohort (from 10xAD cell 12)
tr = pd.read_csv(TAB / "10x_12_transcript_level_astro.csv")
pm = pd.read_csv(TAB / "10x_12_transcript_level_per_microglia.csv", index_col=0)
pm["tg"] = (pm["genotype"] != "WT").astype(int)
dC = pm[((pm["tg"] == 0) & (pm["age_mo"] == 13.4)) | ((pm["tg"] == 1) & (pm["age_mo"] == 17.9))].copy()
inter = []
f = smf.glm("cover ~ spp1 + spp1:tg + C(sample):C(ctx_bin)", data=dC, family=sm.families.Poisson()).fit(cov_type="HC1")
lo, hi = np.exp(f.conf_int().loc["spp1:tg"])
inter.append({"measure": "coverage (homeostatic astrocyte transcripts)", "ratio_of_ratios_Tg_vs_WT": np.exp(f.params["spp1:tg"]), "lo": lo, "hi": hi, "p": f.pvalues["spp1:tg"]})
for g in ["Gfap", "Serpina3n", "C4b", "Aqp4", "Cd44", "Vim", "Osmr"]:
    dd = dC[(dC[g] + dC["cover"]) > 0]
    X = pd.get_dummies(dd["sample"].astype(str) + "|" + dd["ctx_bin"].astype(str), drop_first=True).astype(float)
    X["spp1"] = dd["spp1"]; X["spp1_tg"] = dd["spp1"] * dd["tg"]
    m_ = sm.GLM(np.column_stack([dd[g], dd["cover"]]), sm.add_constant(X), family=sm.families.Binomial()).fit(cov_type="HC1")
    lo, hi = np.exp(m_.conf_int().loc["spp1_tg"])
    inter.append({"measure": g, "ratio_of_ratios_Tg_vs_WT": np.exp(m_.params["spp1_tg"]), "lo": lo, "hi": hi, "p": m_.pvalues["spp1_tg"]})
inter = pd.DataFrame(inter)
share = (dC.assign(gfap_share=dC["Gfap"] / (dC["Gfap"] + dC["cover"]).replace(0, np.nan))
           .groupby(["genotype", "age_mo", "spp1"]).agg(microglia=("cover", "size"), mean_cover=("cover", "mean"),
                                                         mean_gfap_share=("gfap_share", "mean")).reset_index())
keep_cols = [c for c in ["sample", "genotype", "age_mo", "ctx_bin", "d_vlmc", "Spp1_n", "spp1", "cover", "Aldh1l1", "Gja1", "Acsbg1",
                         "Slc39a12", "Ntsr2", "Gfap", "Serpina3n", "C4b", "Cd44", "Vim", "Osmr", "C3", "Aqp4", "x_centroid", "y_centroid"] if c in pm]
write_book(XLS / "Supplementary_Table_S6_transcript_level_astrocytes.xlsx",
           "Supplementary Table S6. Transcript-level astrocyte signal around Spp1+ and Spp1− microglia, public cohort",
           [("A_per_animal", "A: per animal and variant: coverage rate ratio (Poisson) and gene odds ratios relative to homeostatic astrocyte transcripts (binomial), context as fixed effect, HC1, BH q within animal", tr),
            ("B_WT_vs_TgCRND8", "B: interaction of Spp1 state and genotype, WT 13.4 vs TgCRND8 17.9 months (section x context fixed effects); value > 1 = smaller Spp1+ effect under amyloid", inter),
            ("C_group_means", "C: mean homeostatic astrocyte transcripts and mean Gfap share per group (WT 13.4, TgCRND8 17.9)", share),
            ("D_per_microglia", "D: per strict microglia: transcript counts 5 to 25 µm (transcripts assigned to microglia removed, qv ≥ 20)", pm[keep_cols])])

# --- Table S6: source data of the supplementary figures (collected in S1 to S6)
write_book(XLS / "Supplementary_Table_S7_source_data_supplementary_figures.xlsx",
           "Supplementary Table S7. Source data of Supplementary Figures S1 to S3",
           [(k, d, df) for k, (d, df) in SD.items()])
display(inter.round(4))